# Validación de NULLs — Fast and Safe DW

In [17]:
import pandas as pd
import yaml
from sqlalchemy import create_engine

with open('../config.yml', 'r') as f:
    config = yaml.safe_load(f)
cfg = config['TARGET_DB']
url = f"{cfg['drivername']}://{cfg['user']}:{cfg['password']}@{cfg['host']}:{cfg['port']}/{cfg['dbname']}"
dw = create_engine(url)
print('Conexión OK')

Conexión OK


## 1. Verificar que NO existen NULLs en FKs de hecho_servicio

In [18]:
fks = [
    'fk_fecha_solicitud', 'fk_fecha_entrega', 'fk_cliente',
    'fk_sede', 'fk_mensajero', 'fk_ciudad_origen',
    'fk_ciudad_destino', 'fk_tipo_servicio'
]
for fk in fks:
    n = pd.read_sql(f"SELECT COUNT(*) as n FROM hecho_servicio WHERE {fk} IS NULL", dw).iloc[0,0]
    estado = '✅ OK' if n == 0 else f'❌ {n} NULLs'
    print(f'  {fk:30s}: {estado}')

  fk_fecha_solicitud            : ✅ OK
  fk_fecha_entrega              : ✅ OK
  fk_cliente                    : ✅ OK
  fk_sede                       : ✅ OK
  fk_mensajero                  : ✅ OK
  fk_ciudad_origen              : ✅ OK
  fk_ciudad_destino             : ✅ OK
  fk_tipo_servicio              : ✅ OK


## 2. Verificar que NO existen NULLs en FKs de hecho_transiciones_estado

In [19]:
fks_t = [
    'fk_fecha_transicion', 'fk_mensajero', 'fk_cliente',
    'fk_estado_servicio', 'fk_ciudad_origen', 'fk_ciudad_destino', 'fk_sede'
]
for fk in fks_t:
    n = pd.read_sql(f"SELECT COUNT(*) as n FROM hecho_transiciones_estado WHERE {fk} IS NULL", dw).iloc[0,0]
    estado = '✅ OK' if n == 0 else f'❌ {n} NULLs'
    print(f'  {fk:30s}: {estado}')

  fk_fecha_transicion           : ✅ OK
  fk_mensajero                  : ✅ OK
  fk_cliente                    : ✅ OK
  fk_estado_servicio            : ✅ OK
  fk_ciudad_origen              : ✅ OK
  fk_ciudad_destino             : ✅ OK
  fk_sede                       : ✅ OK


## 3. Verificar que las métricas NO tienen NULLs (deben tener 0.001 en su lugar)

In [20]:
metricas = [
    'tiempo_total_minutos', 'tiempo_fase_asignacion_min',
    'tiempo_fase_recogida_min', 'tiempo_fase_entrega_min'
]
for m in metricas:
    n = pd.read_sql(f"SELECT COUNT(*) as n FROM hecho_servicio WHERE {m} IS NULL", dw).iloc[0,0]
    estado = '✅ OK' if n == 0 else f'❌ {n} NULLs'
    print(f'  {m:35s}: {estado}')

  tiempo_total_minutos               : ✅ OK
  tiempo_fase_asignacion_min         : ✅ OK
  tiempo_fase_recogida_min           : ✅ OK
  tiempo_fase_entrega_min            : ✅ OK


## 4. Verificar que el registro desconocido (key=0) existe en cada dimensión

In [21]:
dims = {
    'dim_fecha':     'key_dim_fecha',
    'dim_cliente':   'key_dim_cliente',
    'dim_sede':      'key_dim_sede',
    'dim_mensajero': 'key_dim_mensajero',
    'dim_ciudad':    'key_dim_ciudad',
}
for tabla, key in dims.items():
    n = pd.read_sql(f"SELECT COUNT(*) as n FROM {tabla} WHERE {key} = 0", dw).iloc[0,0]
    estado = '✅ Existe' if n == 1 else '❌ No encontrado'
    print(f'  {tabla:20s} key=0: {estado}')

  dim_fecha            key=0: ✅ Existe
  dim_cliente          key=0: ✅ Existe
  dim_sede             key=0: ✅ Existe
  dim_mensajero        key=0: ✅ Existe
  dim_ciudad           key=0: ✅ Existe


## 5. Ver cuántos servicios apuntan al mensajero desconocido (key=0)

In [22]:
pd.read_sql("""
    SELECT m.nombre, COUNT(*) AS total_servicios
    FROM hecho_servicio h
    JOIN dim_mensajero m ON m.key_dim_mensajero = h.fk_mensajero
    WHERE h.fk_mensajero = 0
    GROUP BY m.nombre
""", dw)

,nombre,total_servicios
0,SIN MENSAJERO ASIGNADO,710


## 6. Verificar que los promedios ya no se ven afectados por NULLs
(Si hay 0.001 en vez de NULL, el COUNT y AVG incluyen todos los registros)

In [24]:
total = pd.read_sql("SELECT COUNT(*) as total FROM hecho_servicio", dw).iloc[0,0]
en_avg = pd.read_sql("SELECT COUNT(*) as n FROM hecho_servicio WHERE tiempo_total_minutos IS NOT NULL", dw).iloc[0,0]
print(f'Total servicios         : {total}')
print(f'Incluidos en el AVG     : {en_avg}')
print(f'Porcentaje cubierto     : {round(en_avg/total*100, 1)}%')
print()
pd.read_sql("""
    SELECT
        ROUND(AVG(NULLIF(tiempo_total_minutos, 0.001))::numeric, 2)       AS promedio_real_min,
        ROUND(AVG(NULLIF(tiempo_fase_asignacion_min, 0.001))::numeric, 2) AS promedio_asignacion_min,
        ROUND(AVG(NULLIF(tiempo_fase_recogida_min, 0.001))::numeric, 2)   AS promedio_recogida_min,
        ROUND(AVG(NULLIF(tiempo_fase_entrega_min, 0.001))::numeric, 2)    AS promedio_entrega_min
    FROM hecho_servicio
    WHERE tiempo_total_minutos > 0.001
""", dw)

Total servicios         : 28328
Incluidos en el AVG     : 28328
Porcentaje cubierto     : 100.0%



,promedio_real_min,promedio_asignacion_min,promedio_recogida_min,promedio_entrega_min
0,527.43,110.05,79.13,338.27
